# Perfume Dataset Cleaning Notebook

**Goal:**  
Clean and preprocess the original perfume dataset (`final_perfume_data.csv`)  
for use in the Perfume Recommendation System.

In [2]:
import pandas as pd
import numpy as np
import re
import os


In [6]:
# Load the original dataset safely, handling special characters
try:
    df = pd.read_csv("final_perfume_data.csv", encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv("final_perfume_data.csv", encoding="ISO-8859-1")

print(" Raw data loaded successfully!")
print("Shape:", df.shape)
df.head()


 Raw data loaded successfully!
Shape: (2191, 5)


,Name,Brand,Description,Notes,Image URL
0,Tihota Eau de Parfum,Indult,"Rapa Nui for sugar, Tihota is, quite simply, ...","Vanilla bean, musks",https://static.luckyscent.com/images/products/...
1,Sola Parfum,Di Ser,A tribute to the expanse of space extending f...,"Lavender, Yuzu, Lemongrass, Magnolia, Geraniu...",https://static.luckyscent.com/images/products/...
2,Kagiroi Parfum,Di Ser,An aromatic ode to the ancient beauty of Japa...,"Green yuzu, green shikuwasa, sansho seed, cor...",https://static.luckyscent.com/images/products/...
3,Velvet Fantasy Eau de Parfum,Montale,Velvet Fantasy is a solar fragrance where cit...,"tangerine, pink pepper, black coffee, leat...",https://static.luckyscent.com/images/products/...
4,A Blvd. Called Sunset Eau de Parfum,A Lab on Fire,There's no way A Lab On Fire could relocate t...,"Bergamot, almond, violet, jasmine, leather, s...",https://static.luckyscent.com/images/products/...


In [8]:
# Inspect structure and missing values
df.info()
df.isnull().sum().sort_values(ascending=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2191 entries, 0 to 2190
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Name         2191 non-null   object
 1   Brand        2191 non-null   object
 2   Description  2191 non-null   object
 3   Notes        2111 non-null   object
 4   Image URL    2191 non-null   object
dtypes: object(5)
memory usage: 85.7+ KB


Notes          80
Name            0
Brand           0
Description     0
Image URL       0
dtype: int64

In [12]:
# Remove duplicate rows
df = df.drop_duplicates()

# Drop irrelevant or redundant columns
drop_cols = ['Unnamed: 0', 'index', 'extra', 'url.1']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

print("Duplicates and unnecessary columns removed")
df.head(2)


Duplicates and unnecessary columns removed


,Name,Brand,Description,Notes,Image URL
0,Tihota Eau de Parfum,Indult,"Rapa Nui for sugar, Tihota is, quite simply, ...","Vanilla bean, musks",https://static.luckyscent.com/images/products/...
1,Sola Parfum,Di Ser,A tribute to the expanse of space extending f...,"Lavender, Yuzu, Lemongrass, Magnolia, Geraniu...",https://static.luckyscent.com/images/products/...


In [14]:
# Function to clean text
def clean_text(x):
    if pd.isna(x):
        return ""
    x = re.sub(r'\s+', ' ', str(x))  # remove extra spaces
    x = re.sub(r'[^a-zA-Z0-9,.\-\' ]', '', x)  # remove unwanted characters
    return x.strip()

# Apply cleaning to common text columns
text_cols = ['Name', 'Brand', 'Notes', 'Description']
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

print("Text fields cleaned")
df[text_cols].head(3)


Text fields cleaned


,Name,Brand,Notes,Description
0,Tihota Eau de Parfum,Indult,"Vanilla bean, musks","Rapa Nui for sugar, Tihota is, quite simply, T..."
1,Sola Parfum,Di Ser,"Lavender, Yuzu, Lemongrass, Magnolia, Geranium...",A tribute to the expanse of space extending fr...
2,Kagiroi Parfum,Di Ser,"Green yuzu, green shikuwasa, sansho seed, cori...",An aromatic ode to the ancient beauty of Japan...


In [16]:
# Convert column names to lowercase and replace spaces
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Column names standardized")
df.head(2)


Column names standardized


,name,brand,description,notes,image_url
0,Tihota Eau de Parfum,Indult,"Rapa Nui for sugar, Tihota is, quite simply, T...","Vanilla bean, musks",https://static.luckyscent.com/images/products/...
1,Sola Parfum,Di Ser,A tribute to the expanse of space extending fr...,"Lavender, Yuzu, Lemongrass, Magnolia, Geranium...",https://static.luckyscent.com/images/products/...


In [18]:
# Fill missing values for key text fields
for col in ['brand', 'notes', 'description']:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# Fill numeric columns if available
if 'rating' in df.columns:
    df['rating'] = df['rating'].fillna(df['rating'].median())

print("Missing values handled")


Missing values handled


In [20]:
# Combine name, brand, and notes for embedding or NLP modeling
if {'name', 'brand', 'notes'}.issubset(df.columns):
    df['text_all'] = df['name'] + " " + df['brand'] + " " + df['notes']

print("Added 'text_all' column combining descriptive features")
df[['name', 'brand', 'notes', 'text_all']].head(3)


Added 'text_all' column combining descriptive features


,name,brand,notes,text_all
0,Tihota Eau de Parfum,Indult,"Vanilla bean, musks","Tihota Eau de Parfum Indult Vanilla bean, musks"
1,Sola Parfum,Di Ser,"Lavender, Yuzu, Lemongrass, Magnolia, Geranium...","Sola Parfum Di Ser Lavender, Yuzu, Lemongrass,..."
2,Kagiroi Parfum,Di Ser,"Green yuzu, green shikuwasa, sansho seed, cori...","Kagiroi Parfum Di Ser Green yuzu, green shikuw..."


In [22]:
# Save the cleaned data
os.makedirs("data", exist_ok=True)
CLEAN_PATH = "data/clean_perfume_data.csv"
df.to_csv(CLEAN_PATH, index=False)

print(f" Cleaned dataset saved to: {CLEAN_PATH}")
print("Final shape:", df.shape)


 Cleaned dataset saved to: data/clean_perfume_data.csv
Final shape: (2191, 6)


In [24]:
# View quick summary stats to confirm cleaning
df.describe(include='all').T.head(10)


,count,unique,top,freq
name,2191,2184,New York Intense Eau de Parfum,2
brand,2191,249,TOM FORD Private Blend,39
description,2191,2165,Bal d'Afrique is inspired by Paris in the late...,2
notes,2191,2054,,80
image_url,2191,2191,https://static.luckyscent.com/images/products/...,1
text_all,2191,2191,"Tihota Eau de Parfum Indult Vanilla bean, musks",1
